# NB04 — Model development on MIMIC-IV

**Purpose.** Train the primary model (LightGBM, hyper-parameters tuned by 5-fold cross-validation) and two comparators (logistic regression, XGBoost) on MIMIC-IV 2008–2016 using the V1 harmonised features. Report internal (out-of-fold) and temporal (2017–2022) validation.

**Research question.** RQ1 (development part)

| | |
|---|---|
| **Inputs** | `data/analysis/mimic_analysis.parquet` |
| **Outputs** | `models/*.pkl`, `models/pred_mimic.parquet`, Table S (internal/temporal performance) |
| **Free-Colab runtime** | 20–40 min |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB04')

In [ ]:
import joblib, json, lightgbm as lgb, xgboost as xgb
from sklearn.model_selection import StratifiedKFold, ParameterSampler, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import log_loss

m, e = tu.load_analysis()
FEATS = tu.feature_set('V1')
dev = m[m['period'] == 'dev'].reset_index(drop=True)
tt = m[m['period'] == 'temporal_test'].reset_index(drop=True)
Xd, yd = dev[FEATS].astype(float), dev['mortality'].astype(int).values
Xt, yt = tt[FEATS].astype(float), tt['mortality'].astype(int).values
print(f'{len(FEATS)} features | dev n={len(dev):,} (mortality {yd.mean():.3f}) | '
      f'temporal test n={len(tt):,} (mortality {yt.mean():.3f})')

### 1. LightGBM: random search with 5-fold CV and early stopping (log-loss)

In [ ]:
SPACE = dict(num_leaves=[15, 31, 63, 127], learning_rate=[0.02, 0.05, 0.1],
             min_child_samples=[20, 50, 100, 200], subsample=[0.7, 0.85, 1.0],
             colsample_bytree=[0.5, 0.7, 0.9], reg_lambda=[0, 1, 5, 10], max_depth=[-1, 6, 10])
N_SEARCH = 4 if tu.FAST else 25
skf = StratifiedKFold(5, shuffle=True, random_state=tu.SEED)
folds = list(skf.split(Xd, yd))

def cv_lgbm(params):
    losses, iters = [], []
    for tr, va in folds:
        mdl = lgb.LGBMClassifier(n_estimators=3000, subsample_freq=1, random_state=tu.SEED,
                                 verbose=-1, n_jobs=-1, **params)
        mdl.fit(Xd.iloc[tr], yd[tr], eval_set=[(Xd.iloc[va], yd[va])], eval_metric='binary_logloss',
                callbacks=[lgb.early_stopping(100, verbose=False)])
        p = mdl.predict_proba(Xd.iloc[va], num_iteration=mdl.best_iteration_)[:, 1]
        losses.append(log_loss(yd[va], p)); iters.append(mdl.best_iteration_)
    return float(np.mean(losses)), int(np.mean(iters))

def search():
    rows = []
    for i, params in enumerate(ParameterSampler(SPACE, N_SEARCH, random_state=tu.SEED)):
        loss, it = cv_lgbm(params)
        rows.append(dict(params, cv_logloss=loss, n_estimators=it))
        print(f'  config {i+1}/{N_SEARCH}: logloss={loss:.4f}  trees={it}', flush=True)
    return pd.DataFrame(rows).sort_values('cv_logloss')

search_df = tu.cached('lgbm_search', search)
tu.save_table(search_df.head(10), 'lgbm_search_top10')
best = search_df.iloc[0].to_dict()
BEST = {k: (int(best[k]) if k in ('num_leaves', 'min_child_samples', 'max_depth') else float(best[k]))
        for k in SPACE}
BEST_NTREES = int(best['n_estimators'])
print('BEST:', BEST, 'trees:', BEST_NTREES)
json.dump(dict(params=BEST, n_estimators=BEST_NTREES), open(os.path.join(tu.MODELS, 'lgbm_best_params.json'), 'w'))

In [ ]:
def make_lgbm():
    return lgb.LGBMClassifier(n_estimators=BEST_NTREES, subsample_freq=1, random_state=tu.SEED,
                              verbose=-1, n_jobs=-1, **BEST)

# out-of-fold predictions (internal validation) and final model on all development data
oof_lgbm = np.zeros(len(dev))
for tr, va in folds:
    oof_lgbm[va] = make_lgbm().fit(Xd.iloc[tr], yd[tr]).predict_proba(Xd.iloc[va])[:, 1]
lgbm = make_lgbm().fit(Xd, yd)
joblib.dump(lgbm, os.path.join(tu.MODELS, 'lgbm_V1.pkl'))
print('LightGBM saved.')

### 2. Logistic regression (median imputation + standardisation inside the pipeline; C by 5-fold CV)

In [ ]:
lrcv = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                 ('lr', LogisticRegressionCV(Cs=10, cv=5, scoring='neg_log_loss', max_iter=5000,
                                             random_state=tu.SEED))]).fit(Xd, yd)
C_BEST = float(lrcv.named_steps['lr'].C_[0])
def make_lr():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('lr', LogisticRegression(C=C_BEST, max_iter=5000))])
oof_lr = np.zeros(len(dev))
for tr, va in folds:
    oof_lr[va] = make_lr().fit(Xd.iloc[tr], yd[tr]).predict_proba(Xd.iloc[va])[:, 1]
lr = make_lr().fit(Xd, yd)
joblib.dump(lr, os.path.join(tu.MODELS, 'lr_V1.pkl'))
print('LR saved. C =', C_BEST)

### 3. XGBoost (supplementary comparator; trees chosen by early stopping on a 15% split)

In [ ]:
Xa, Xv, ya, yv = train_test_split(Xd, yd, test_size=0.15, stratify=yd, random_state=tu.SEED)
xg0 = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=5, subsample=0.8,
                        colsample_bytree=0.7, min_child_weight=5, tree_method='hist',
                        eval_metric='logloss', early_stopping_rounds=100, random_state=tu.SEED, n_jobs=-1)
xg0.fit(Xa, ya, eval_set=[(Xv, yv)], verbose=False)
XGB_TREES = int(xg0.best_iteration) + 1
def make_xgb():
    return xgb.XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=5, subsample=0.8,
                             colsample_bytree=0.7, min_child_weight=5, tree_method='hist',
                             eval_metric='logloss', random_state=tu.SEED, n_jobs=-1)
oof_xgb = np.zeros(len(dev))
for tr, va in folds:
    oof_xgb[va] = make_xgb().fit(Xd.iloc[tr], yd[tr]).predict_proba(Xd.iloc[va])[:, 1]
xgbm = make_xgb().fit(Xd, yd)
joblib.dump(xgbm, os.path.join(tu.MODELS, 'xgb_V1.pkl'))
json.dump(dict(C_lr=C_BEST, xgb_trees=XGB_TREES), open(os.path.join(tu.MODELS, 'other_params.json'), 'w'))
print('XGB saved. trees =', XGB_TREES)

### 4. Internal (out-of-fold) and temporal validation with bootstrap 95% CIs

In [ ]:
models = {'LightGBM': lgbm, 'Logistic regression': lr, 'XGBoost': xgbm}
oof = {'LightGBM': oof_lgbm, 'Logistic regression': oof_lr, 'XGBoost': oof_xgb}
pt = {k: mdl.predict_proba(Xt)[:, 1] for k, mdl in models.items()}
rows = []
for k in models:
    for setname, y_, p_ in [('MIMIC internal (5-fold OOF)', yd, oof[k]), ('MIMIC temporal test 2017-2022', yt, pt[k])]:
        ci, _ = tu.bootstrap(y_, p_, B=tu.n_boot(500))
        for _, r in ci.iterrows():
            rows.append(dict(model=k, dataset=setname, metric=r['metric'], estimate=r['estimate'],
                             lo=r['lo'], hi=r['hi']))
perf = pd.DataFrame(rows)
tu.save_table(perf.round(4), 'internal_temporal_performance', show=False)
wide = perf.assign(txt=perf.apply(lambda r: tu.fmt_ci(r.estimate, r.lo, r.hi), axis=1)) \
           .pivot_table(index=['model', 'dataset'], columns='metric', values='txt', aggfunc='first')
tu.save_table(wide[['auroc', 'auprc', 'brier', 'oe', 'citl', 'slope', 'ici']], 'table_internal_temporal', index=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
tu.plot_calibration(axes[0], {k: (yd, oof[k]) for k in models}, 'MIMIC internal (OOF)', hist_p=oof_lgbm)
tu.plot_calibration(axes[1], {k: (yt, pt[k]) for k in models}, 'MIMIC temporal test 2017-2022', hist_p=pt['LightGBM'])
tu.save_fig(fig, 'fig_calibration_internal_temporal'); plt.show()

imp = pd.DataFrame(dict(feature=FEATS, gain=lgbm.booster_.feature_importance('gain')))
imp['gain_pct'] = 100 * imp['gain'] / imp['gain'].sum()
imp = imp.sort_values('gain_pct', ascending=False)
tu.save_table(imp.head(20).round(2), 'lgbm_feature_importance_top20')

In [ ]:
pred_m = pd.concat([
    pd.DataFrame(dict(stay_id=dev['stay_id'], set='dev_oof', mortality=yd, p_lgbm=oof_lgbm, p_lr=oof_lr, p_xgb=oof_xgb,
                      anchor_year_group=dev['anchor_year_group'])),
    pd.DataFrame(dict(stay_id=tt['stay_id'], set='temporal_test', mortality=yt, p_lgbm=pt['LightGBM'],
                      p_lr=pt['Logistic regression'], p_xgb=pt['XGBoost'], anchor_year_group=tt['anchor_year_group']))])
pred_m.to_parquet(os.path.join(tu.MODELS, 'pred_mimic.parquet'), index=False)

tu.finish_notebook({
    'n_dev': len(dev), 'n_temporal_test': len(tt), 'n_features': len(FEATS),
    'lgbm_best_params': BEST, 'lgbm_trees': BEST_NTREES, 'lr_C': C_BEST, 'xgb_trees': XGB_TREES,
    'performance': perf.round(4).to_dict('records'),
    'top10_features': imp.head(10)[['feature', 'gain_pct']].round(2).to_dict('records'),
})